In [ ]:
import gymnasium as gym
from gymnasium_env import BlokusEnv
from src.agents.agent import Agent, QL_Agent
import numpy as np
from tqdm import tqdm
import pickle
import cProfile

In [ ]:
import random

q_table = {}
def encode_board(board: np.ndarray) -> str:
    return ''.join(map(str, board.flatten()))

def decode_board(encoded_board : str, n: int) -> np.ndarray:
    board = []
    board = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            board[i, j] = int(encoded_board[i * n + j])
    return board

def create_q_table(state: str, actions: np.ndarray):
    if state not in q_table:
        q_table[state] = {action: 0 for action in actions}
    

def get_q_value(state: str, action: int) -> float:
    return q_table[state][action]

def set_q_value(state: str, action: int, value: float):
    q_table[state][action] = value

not_visited = 0
def argmax(state: str) -> int:
    if not state in q_table:
        global not_visited
        not_visited += 1
        return -1
    return max(q_table[state], key=q_table[state].get)

def maxi(state: str) -> float:
    if len(q_table[state].values()) == 0:
        return 0
    return max(q_table[state].values())

# env.render_mode = "console"
with open('q_table1.pkl', 'rb') as f:
    q_table = pickle.load(f)



In [ ]:
class SingleAgentBlokusEnv(BlokusEnv):
    AGENT_INDEX = 1
    def __init__(self, hidden_agent = QL_Agent(name="hey", dictionary="/Users/mario/Documents/proj/cam/Blokus/notebooks/q_table2.pkl"),  *args, **kwargs, ):
        self.agent = hidden_agent
        super(SingleAgentBlokusEnv, self).__init__(*args, **kwargs)
        
    def step(self, action):
        assert self.current_player == 2
        # assert False # check this!!! There is a bug here

        obs, total_reward, terminated, truncated, info = super(SingleAgentBlokusEnv, self).step(action)
        if terminated or truncated: 
            return obs, total_reward, terminated, truncated, info
        
        while self.current_player == self.AGENT_INDEX:
            actions = self.possible_actions(self.AGENT_INDEX) # get possible actions for the random agent
            assert len(actions) > 0
            action = self.agent.get_action(obs)
            if action is None:
                action = random.choice(actions)
            obs, reward, terminated, truncated, info = super(SingleAgentBlokusEnv, self).step(action)
            total_reward -= reward
            assert not truncated
            if terminated:
                return obs, total_reward, terminated, truncated, info
        
        return obs, total_reward, terminated, truncated, info

    def reset(self, *args, **kwargs):
        obs1, info1 = super(SingleAgentBlokusEnv, self).reset(*args, **kwargs)
        actions = self.possible_actions(self.AGENT_INDEX) # get possible actions for the random agent
        assert len(actions) > 0
        action = self.agent.get_action(obs1)
        obs, reward, terminated, truncated, info = super(SingleAgentBlokusEnv, self).step(action)
        return obs, info

# Create an instance of the custom environment

In [ ]:
env = SingleAgentBlokusEnv(board_size=7, num_players=2, render_mode="human", render_scale=10, mode = "good")

In [ ]:

# Create an instance of the custom environment
def test_agent(env, num_episodes=1):
    global q_table
    win_counter, tie_counter, lose_counter = 0, 0, 0
    for _ in range(num_episodes):
        obs, info = env.reset()
        # state = encode_board(obs["state"])
        done = False
        total_reward = 0

        while not done:
            print(f"----------------------------------------------")
            print(f"Possible actions:")
            for idx, i in enumerate(list(map(env._action_to_tuple, obs['possible_actions']))):
                print(idx, ":", i)
            action = obs['possible_actions'][int(input("Enter action: "))]
            
            next_obs, reward, terminated, truncated, info = env.step(action)
            # next_state = encode_board(next_obs["state"])
            # state = next_state
            obs = next_obs
            total_reward += reward
            done = terminated or truncated
            # env.render()
        if total_reward > 0:
            win_counter += 1
        elif total_reward == 0:
            tie_counter += 1
        else:
            lose_counter += 1

    return win_counter, tie_counter, lose_counter

# win_counter1 = np.zeros(1000)
# tie_counter1 = np.zeros(1000)
# lose_counter1 = np.zeros(1000)
# # Test the agent
# for i in tqdm(range(1000)):
#     win_counter1[i], tie_counter1[i], lose_counter1[i] = test_agent(env)


# print(f"Number of not visited states: {not_visited}")

test_agent(env, 1)